# Pipeline stage visualization

Static replacement for the former `debug_app.py` Tk GUI. Runs the pipeline **once**
on a single page, then for every stage renders:

- one **original page** image, then
- for each overlay category of that stage, a pair: the category's geometry drawn
  alone on white (**isolated**), and the original page with that geometry drawn on
  top (**overlay**).

So a single-overlay stage produces 3 images; a stage with *N* overlays produces
`1 + 2N`. `clustering` and `color_separation` expand **every** category / bucket,
which can be 100+ images on a dense page.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1] if (Path.cwd() / "pipeline_stage_visualization.ipynb").exists() else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from rastervec.logging_setup import configure_logging
from rastervec.paths import output_dir
from rastervec.pipelines.current import STEP_NAMES, run_pipeline
from rastervec.renderer import render_reconstructed_pdf
from rastervec.renderer.notebook import RenderResult, page_setup, show_row, visualize
from rastervec.renderer.stages import (
    render_clustering_steps,
    render_drawing,
    render_fast,
    render_layer_color_buckets,
    render_layers,
    render_native,
    render_ocr_results,
    render_radon,
    render_restore,
    render_similarity,
    render_text_candidates,
    render_vectors,
)

configure_logging()

## Parameters

`VARIANT` picks one of the `current` pipeline variants
(`Evaluation/Evaluate/variants.py`: `current`, `current_nofast`) — it sets
`enable_fast` exactly as the benchmark runs it. The `legacy` variant is not
runnable here (it's `archive/raster_parser`, not this pipeline).

In [ ]:
from rastervec.Evaluation.Evaluate.variants import VARIANTS, resolve_variant

PDF_PATH = next(iter(sorted((PROJECT_ROOT / "references").glob("*.pdf"))), None)
PAGE_INDEX = 3
ZOOM = 2.0

# Which pipeline variant to visualize. Only "current"-engine variants run here
# (legacy is archive/raster_parser, not this pipeline): "current", "current_nofast".
# The variant sets enable_fast below.
VARIANT = "current"

# Pool 2 (compute) workers: FAST tile detection + OCR crop recognition are
# dispatched into a shared process pool instead of running inline. 0 = fully
# local (default). Pool 1 / page-level parallelism does not apply here -- this
# notebook runs a single page -- so only the compute pool is configurable.
COMPUTE_WORKERS = 0

_variant = resolve_variant(VARIANT)
assert _variant.engine == "current", (
    f"{VARIANT!r} is engine={_variant.engine!r}; only 'current' variants run in this notebook. "
    f"current variants: {[n for n, v in VARIANTS.items() if v.engine == 'current']}"
)
ENABLE_FAST = _variant.enable_fast

assert PDF_PATH is not None, "no PDF under references/ -- set PDF_PATH by hand"
print("PDF:", PDF_PATH, "| page", PAGE_INDEX)
print(f"variant: {_variant.name}  (enable_fast={_variant.enable_fast})")

## Run the pipeline

In [ ]:
from rastervec.Reader.Parallel import compute_pool

with compute_pool(COMPUTE_WORKERS) as _compute:
    res = run_pipeline(
        str(PDF_PATH), PAGE_INDEX,
        enable_fast=_variant.enable_fast, verbose=True, compute=_compute,
    )
outputs = res.step_outputs or {}
page = res.page
assert page is not None, "reader step failed"

for name, o in outputs.items():
    print(f"{'ok ' if o.status == 'ok' else 'ERR'}  {name:10}  {o.error or ''}")


## Setup

In [ ]:
# Every render_<stage_name> function now lives in rastervec.renderer.stages
# (see that module's docstring) and reads `res` directly; this is the one
# bit of shared setup they all need: the rasterized original page at ZOOM,
# plus the rotation-baked display matrix.
ORIGINAL, MATRIX = page_setup(res, ZOOM)

RECON_DIR = output_dir("pipeline_stage_visualization")

# Only these stages write a page-space bbox-overlay PDF (via export_path=
# below): clustering, the classify text-candidates sub-step, segment
# (Radon), FAST, and both OCR layers (passed/failed + restored). Every
# other stage's visualize(...) call still renders its inline images, just
# without writing a PDF -- stage_pdf_path returns None for a label not in
# this set, and visualize() already treats export_path=None as "skip the
# PDF write". The final full-page reconstruction cell writes its own PDF
# separately (not gated by this set).
PDF_STAGES = {
    "classify_clustering_steps", "classify_text_candidates",
    "segment", "fast", "ocr", "restore",
}


def stage_pdf_path(label: str) -> "Path | None":
    if label not in PDF_STAGES:
        return None
    return RECON_DIR / f"{PDF_PATH.stem}_p{PAGE_INDEX}_{label}_boxes.pdf"

## 1. Reader

In [ ]:
visualize(
    "read",
    RenderResult(note=(
        f"mediabox={page.meta.mediabox}  rotation={page.meta.rotation}  "
        f"size={page.meta.width:.0f}x{page.meta.height:.0f}"
    )),
    step_outputs=outputs, original=ORIGINAL, matrix=MATRIX,
)

## 2. Native Text

In [ ]:
visualize("native", render_native(res, zoom=ZOOM), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("native"))

## 3. Vector Extraction

In [ ]:
visualize("vectors", render_vectors(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("vectors"))

## 4. Layer + Color Separation  (inside `classify`)


In [ ]:
visualize("classify", render_layers(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("classify_layers"))

In [ ]:
visualize("classify", render_layer_color_buckets(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("classify_color_buckets"))

## 5. Clustering  (the 12-step chain, per bucket)


In [ ]:
visualize("classify", render_clustering_steps(res, MATRIX, ORIGINAL), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("classify_clustering_steps"))

## 6. Text Candidates  (`classify`)

In [ ]:
visualize("classify", render_text_candidates(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("classify_text_candidates"))

## 7. Similarity

Groups clusters by whole-page, translation+rotation-tolerant shape
equivalence, using a PCA principal-axis estimate over each cluster's own
point cloud -- pure vector-geometry math, no rendering, since this now
runs *before* Radon (see `pipelines/_steps.py::_cluster_angle`,
`build_cluster_candidates`). Each group beyond size 1 is a dedup
opportunity: only one member (the representative) needs Radon segmentation
and OCR at all (Phase G/H below); the rest get their words' `Text` readings
restored from it (Phase I).

In [ ]:
visualize("similarity", render_similarity(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("similarity"))

## 8. FAST: Text Detect

Scores every cluster candidate at its real page position against a
whole-page FAST render; a similarity group passes only if *every* member
individually exceeds `FAST_COMBINED_KEEP_THRESHOLD`. A passing group
materializes one `UniqueSegment` -- its representative's canonical
vectors, a *whole cluster*, not yet Radon-segmented into words; a failing
group's real vectors fold into drawing output.

In [ ]:
fr = res.fast_result
if fr is not None:
    for gi, score in sorted((fr.scores or {}).items()):
        print(f"  group {gi:3}  combined score {score * 100:5.1f}%")
print(f"{len(res.unique_segments or [])} representative cluster(s) passed, {len(res.fast_dropped_vectors or [])} vector(s) dropped to drawing")
visualize("fast", render_fast(res, enable_fast=ENABLE_FAST), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("fast"))

## 9. Segment (Radon)

Runs *after* similarity grouping and FAST detection have already deduped
clusters down to one elected representative per group -- only those
representatives are rendered and Radon-segmented into word-level
`Segment`s (with each word's own deskewed crop captured directly, see
`OCR/radon.py`'s docstring), not every surviving classification cluster.

In [ ]:
visualize("segment", render_radon(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("segment"))

## 10. PaddleOCR

Only the words of representative clusters are recognized here -- no
render happens in this step, since Radon (above) already captured each
word's deskewed crop directly. One canonical-frame `Text` per word,
single recognition pass with PaddleOCR's own angle classifier resolving
the 0/180 flip Radon can't.

In [ ]:
for words in (res.unique_texts or []):
    for t in words:
        if t.text.strip():
            print(f"  {t.text!r:42}  conf={t.confidence:.2f}  angle={t.angle():>7.2f}")
visualize("ocr", render_ocr_results(res, zoom=ZOOM), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("ocr"))

## 11. Restore

For every real cluster occurrence recorded in `SegmentMeta`, replays every
one of its representative's word-level `Text`s through the same
transform, geometrically placing them back onto that occurrence's real
page position/rotation -- the dedup payoff made visible.

In [ ]:
visualize("restore", render_restore(res), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("restore"))

## 12. Drawing Vectors

In [ ]:
visualize("drawing", render_drawing(res, zoom=ZOOM), step_outputs=outputs, original=ORIGINAL, matrix=MATRIX, page_meta=res.page.meta, export_path=stage_pdf_path("drawing"))

## 13. Final Reconstruction

Combines every captured element -- native text, drawing vectors, and restored
OCR text -- into one full-page reconstruction: written as a real
(selectable-text) PDF to `outputs/pipeline_stage_visualization/`, plus an
inline raster preview alongside the original page.

In [ ]:
import io

import pymupdf as fitz
from PIL import Image

native_words = [t for t in res.texts if t.source == "native"]
ocr_results = [t for t in res.texts if t.source == "ocr"]

recon_bytes = render_reconstructed_pdf(
    res.page.meta,
    native_words=native_words,
    drawing_vectors=res.vectors,
    ocr_results=ocr_results,
)
recon_path = RECON_DIR / f"{PDF_PATH.stem}_p{PAGE_INDEX}_reconstructed.pdf"
recon_path.write_bytes(recon_bytes)
print(f"Wrote full reconstruction to {recon_path}")

# Preview built from the same bytes just written, instead of a second
# render_reconstructed_page() call that would rebuild the whole
# reconstruction (every native word + drawing vector + restored OCR word)
# from scratch a second time.
with fitz.open(stream=recon_bytes, filetype="pdf") as _recon_doc:
    pixmap = _recon_doc[0].get_pixmap(matrix=fitz.Matrix(ZOOM, ZOOM), alpha=False)
    preview = Image.open(io.BytesIO(pixmap.tobytes("png")))
    preview.load()
show_row([ORIGINAL, preview], ["original page", "full reconstruction"])

## 14. Timeline

A waterfall of `res.step_durations` -- since the 10 steps run strictly
sequentially, each step's start offset is the cumulative sum of the prior
steps' own durations. Bars are colored by `res.step_outputs[name].status`
(green = ok, red = error) when available.

In [ ]:
import matplotlib.pyplot as plt

names = list(res.step_durations.keys())
widths = list(res.step_durations.values())
starts = [sum(widths[:i]) for i in range(len(names))]
status_by_name = {n: o.status for n, o in (res.step_outputs or {}).items()}
colors = ["#059669" if status_by_name.get(n, "ok") == "ok" else "#dc2626" for n in names]

fig, ax = plt.subplots(figsize=(10, 0.4 * len(names) + 1))
ax.barh(names, widths, left=starts, color=colors)
ax.invert_yaxis()  # first step on top
for name, start, width in zip(names, starts, widths):
    ax.text(start + width / 2, name, f"{width:.2f}s", va="center", ha="center", fontsize=8, color="white")
ax.set_xlabel("elapsed time (s)")
ax.set_title(f"Pipeline step timeline -- total {sum(widths):.2f}s")
plt.tight_layout()
plt.show()

## Reading the results

- Everything a step **drops** (`classify` side categories, `fast` dropped
  vectors) is folded into the final `vectors` output -- the pipeline's "this
  is drawing content, not text" verdict.
- `similarity` groups whole clusters, and `fast` dedups them, both *before*
  `segment` (Radon) and `ocr` run, so a repeated page element (e.g. a
  dimension label appearing 5 times) costs one Radon segmentation and one
  OCR call instead of 5 -- see `restore` for where those 5 real-position
  `Text`s come back out.
- The pipeline always runs all 10 steps (including PaddleOCR); the first run
  downloads the PaddleOCR weights. `enable_fast` comes from the chosen
  `VARIANT` (`current` = on, `current_nofast` = pass-through).